In [1]:
##modules
#%matplotlib widget
#%matplotlib inline
#
%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

#matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf
import numpy as np

import pandas as pd


import pickle


In [5]:

try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_SELF import get_paths_SELF

# Parámetros editables
disco = "g"
layer_script = "event"
subj = "s01b"


# Generar variables automáticamente
path_dict = get_paths_SELF(disco=disco, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")



✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\ICA_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_matlab_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\evoked_event
✅ Carpeta creada: g:\PROYECTO_SELF\channels_structure
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event\raw_hsp
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event\fwd
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event\inverse
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\analysis_event\acw_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\analysis_event\PLE_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\analysis_event\ISC_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\anal

In [6]:

layer_script= "event"  # Cambia esto si es necesario
pickle_file = epochs_clean_path / f"dict_conditions.pkl"
# Cargar el pickle
with open(pickle_file, "rb") as f:
    dict_conditions = pickle.load(f)

#epochs
combinaciones = list(dict_conditions.keys())
print(f"combinaciones: {combinaciones}")

combinacion= combinaciones[0]  # Selecciona la primera combinación

subjects = sorted({f.name.split("_")[0].lower() for f in data_task_edf.glob("*.edf")})
print(f"subj: {subjects}")


combinaciones: ['self_pos', 'self_neu', 'self_neg', 'friend_pos', 'friend_neu', 'friend_neg', 'unk_pos', 'unk_neu', 'unk_neg']
subj: ['s01b', 's02b', 's03b', 's04b', 's05b', 's06b', 's07b', 's08b', 's09b', 's10b', 's11b', 's12b', 's13b', 's14b', 's15b', 's16b', 's17b', 's18b', 's19b', 's20b', 's21b', 's22b', 's23b', 's24b', 's25b', 's26b', 's27b', 's28b', 's29b']


# CANALES 

In [8]:




# Diccionario para almacenar los nombres de canales de cada sujeto
channel_names_by_subject = {}



# Iterar sobre cada sujeto
for subj in subjects:
    try:
        # Cargar las épocas del sujeto
        
        
        edf_file = data_task_edf / f"{subj}_vis_c_BVica-export.edf"
        elp_file = data_task_edf / f"{subj}_vis_c_BVica-export.elp"
        
        # Leer el EDF
        raw = mne.io.read_raw_edf(edf_file, preload=True)
        
        channel_names_by_subject[subj] = set(raw.pick("eeg").ch_names)
        del raw
        print(f"✅ Datos cargados correctamente para el sujeto {subj}")

    except:
        print(f"❌ Error al cargar archivos de {subj}")
        continue

# Comparar los canales entre sujetos
all_subjects = list(channel_names_by_subject.keys())
first_subject = all_subjects[0]  # Tomamos el primer sujeto como referencia
reference_channels = channel_names_by_subject[first_subject]

# Verificar si todos los sujetos tienen los mismos canales
for subj in all_subjects:
    different_channels = reference_channels.symmetric_difference(channel_names_by_subject[subj])  # Canales que difieren
    if different_channels:
        print(f"⚠️ Diferencias encontradas en el sujeto {subj}: {different_channels}")

        # Identificar qué canales están solo en este sujeto y no en otros
        only_in_this_subject = channel_names_by_subject[subj] - reference_channels
        if only_in_this_subject:
            print(f"✅ Canales que solo aparecen en {subj}: {only_in_this_subject}")

        # Identificar qué canales están en otros sujetos pero no en este
        missing_channels = reference_channels - channel_names_by_subject[subj]
        if missing_channels:
            print(f"❌ Canales que faltan en {subj} (están en otros sujetos): {missing_channels}")

print("✅ Comparación completada.")

import pickle
import os

with open(os.path.join(output_analysis, "channel_names_by_subject.pkl"), "wb") as f:
    pickle.dump(channel_names_by_subject, f)
from collections import defaultdict


# Convertir listas a sets
sets_by_subject = {subj: set(chs) for subj, chs in channel_names_by_subject.items()}

# Todos los canales presentes en al menos un sujeto
all_channels = set().union(*sets_by_subject.values())

# Intersección solo si hay al menos un sujeto
subject_sets = list(sets_by_subject.values())
if subject_sets:
    common_channels = subject_sets[0].intersection(*subject_sets[1:])
else:
    common_channels = set()

# Canales que no están en todos los sujetos
non_common_channels = all_channels - common_channels

# Para cada canal no común, ver en qué sujetos aparece
channel_presence = defaultdict(list)
for channel in non_common_channels:
    for subj, ch_set in sets_by_subject.items():
        if channel in ch_set:
            channel_presence[channel].append(subj)

# Mostrar resultados
print(f"✅ Canales presentes en TODOS los sujetos ({len(common_channels)}):")
print(sorted(common_channels))

print(f"\n⚠️ Canales NO presentes en todos los sujetos ({len(non_common_channels)}):")
for channel in sorted(non_common_channels):
    print(f"- {channel} está en: {channel_presence[channel]}")


Extracting EDF parameters from F:\WORKAREA\Datos SELF\Self_Exp2\Exp2_review\Visual_Corregidos\ICA\EDF\s01b_vis_c_BVica-export.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 2465499  =      0.000 ...  4930.998 secs...
✅ Datos cargados correctamente para el sujeto s01b
Extracting EDF parameters from F:\WORKAREA\Datos SELF\Self_Exp2\Exp2_review\Visual_Corregidos\ICA\EDF\s02b_vis_c_BVica-export.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 2289499  =      0.000 ...  4578.998 secs...
✅ Datos cargados correctamente para el sujeto s02b
Extracting EDF parameters from F:\WORKAREA\Datos SELF\Self_Exp2\Exp2_review\Visual_Corregidos\ICA\EDF\s03b_vis_c_BVica-export.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 2242999  =      0.000 ...  4485.998 secs...
✅ Datos cargados correctamente para el sujeto s03b
Extracting EDF parameters

# CANALES VISUALES

In [11]:
##subjects visual
subjects=[]
for subdirectorio in general_datadir.iterdir():
    # Comprobamos que el elemento sea un directorio y que su nombre comience con 'sub-A2'
    if subdirectorio.is_dir() and subdirectorio.name.startswith('sub-V1'):
        # Añadimos el nombre del sujeto a la lista
        subjects.append(subdirectorio.name)
        print(subjects)

subjects

['sub-V1001']
['sub-V1001', 'sub-V1002']
['sub-V1001', 'sub-V1002', 'sub-V1003']
['sub-V1001', 'sub-V1002', 'sub-V1003', 'sub-V1004']
['sub-V1001', 'sub-V1002', 'sub-V1003', 'sub-V1004', 'sub-V1005']
['sub-V1001', 'sub-V1002', 'sub-V1003', 'sub-V1004', 'sub-V1005', 'sub-V1006']
['sub-V1001', 'sub-V1002', 'sub-V1003', 'sub-V1004', 'sub-V1005', 'sub-V1006', 'sub-V1007']
['sub-V1001', 'sub-V1002', 'sub-V1003', 'sub-V1004', 'sub-V1005', 'sub-V1006', 'sub-V1007', 'sub-V1008']
['sub-V1001', 'sub-V1002', 'sub-V1003', 'sub-V1004', 'sub-V1005', 'sub-V1006', 'sub-V1007', 'sub-V1008', 'sub-V1009']
['sub-V1001', 'sub-V1002', 'sub-V1003', 'sub-V1004', 'sub-V1005', 'sub-V1006', 'sub-V1007', 'sub-V1008', 'sub-V1009', 'sub-V1010']
['sub-V1001', 'sub-V1002', 'sub-V1003', 'sub-V1004', 'sub-V1005', 'sub-V1006', 'sub-V1007', 'sub-V1008', 'sub-V1009', 'sub-V1010', 'sub-V1011']
['sub-V1001', 'sub-V1002', 'sub-V1003', 'sub-V1004', 'sub-V1005', 'sub-V1006', 'sub-V1007', 'sub-V1008', 'sub-V1009', 'sub-V1010', 

['sub-V1001',
 'sub-V1002',
 'sub-V1003',
 'sub-V1004',
 'sub-V1005',
 'sub-V1006',
 'sub-V1007',
 'sub-V1008',
 'sub-V1009',
 'sub-V1010',
 'sub-V1011',
 'sub-V1012',
 'sub-V1013',
 'sub-V1015',
 'sub-V1016',
 'sub-V1017',
 'sub-V1019',
 'sub-V1020',
 'sub-V1022',
 'sub-V1024',
 'sub-V1025',
 'sub-V1026',
 'sub-V1027',
 'sub-V1028',
 'sub-V1029',
 'sub-V1030',
 'sub-V1031',
 'sub-V1032',
 'sub-V1033',
 'sub-V1034']

In [23]:



# Diccionario para almacenar los nombres de canales de cada sujeto
channel_names_by_subject_visual = {}



# Iterar sobre cada sujeto
for subj in subjects:
    try:
        # Cargar las épocas del sujeto
        raw = mne.io.read_raw_ctf(pathjoin(general_datadir,f"{subj}", "meg",f"{subj}_task-visual_meg.ds"))        # Obtener la lista de canales del sujeto
        channel_names_by_subject_visual[subj] = set(raw.pick("mag").ch_names)
        del raw
        print(f"✅ Datos cargados correctamente para el sujeto {subj}")

    except:
        print(f"❌ Error al cargar archivos de {subj}")
        continue

# Comparar los canales entre sujetos
all_subjects = list(channel_names_by_subject_visual.keys())
first_subject = all_subjects[0]  # Tomamos el primer sujeto como referencia
reference_channels = channel_names_by_subject_visual[first_subject]

# Verificar si todos los sujetos tienen los mismos canales
for subj in all_subjects:
    different_channels = reference_channels.symmetric_difference(channel_names_by_subject_visual[subj])  # Canales que difieren
    if different_channels:
        print(f"⚠️ Diferencias encontradas en el sujeto {subj}: {different_channels}")

        # Identificar qué canales están solo en este sujeto y no en otros
        only_in_this_subject = channel_names_by_subject_visual[subj] - reference_channels
        if only_in_this_subject:
            print(f"✅ Canales que solo aparecen en {subj}: {only_in_this_subject}")

        # Identificar qué canales están en otros sujetos pero no en este
        missing_channels = reference_channels - channel_names_by_subject_visual[subj]
        if missing_channels:
            print(f"❌ Canales que faltan en {subj} (están en otros sujetos): {missing_channels}")

print("✅ Comparación completada.")


ds directory : g:\MOUS_204\sub-V1001\meg\sub-V1001_task-visual_meg.ds
    res4 data read.
    hc data read.
    Separate EEG position data file not present.
    Quaternion matching (desired vs. transformed):
      -0.50   80.15    0.00 mm <->   -0.50   80.15    0.00 mm (orig :  -71.73   42.77 -259.14 mm) diff =    0.000 mm
       0.50  -80.15    0.00 mm <->    0.50  -80.15    0.00 mm (orig :   37.79  -74.15 -264.78 mm) diff =    0.000 mm
     108.63    0.00    0.00 mm <->  108.63   -0.00    0.00 mm (orig :   62.71   58.11 -264.02 mm) diff =    0.000 mm
    Coordinate transformations established.
    Polhemus data for 3 HPI coils added
    Device coordinate locations for 3 HPI coils added
Picked positions of 4 EEG channels from channel info
    4 EEG locations added to Polhemus data.
    Measurement info composed.
Finding samples for g:\MOUS_204\sub-V1001\meg\sub-V1001_task-visual_meg.ds\sub-V1001_task-visual_meg.meg4: 
    System clock channel is available, checking which samples are v

In [ ]:
import pickle
import os

with open(os.path.join(output_analysis, "channel_names_by_subject_visual.pkl"), "wb") as f:
    pickle.dump(channel_names_by_subject_visual, f)
from collections import defaultdict


In [25]:

# Convertir listas a sets
sets_by_subject = {subj: set(chs) for subj, chs in channel_names_by_subject_visual.items()}

# Todos los canales presentes en al menos un sujeto
all_channels = set().union(*sets_by_subject.values())

# Intersección solo si hay al menos un sujeto
subject_sets = list(sets_by_subject.values())
if subject_sets:
    common_channels = subject_sets[0].intersection(*subject_sets[1:])
else:
    common_channels = set()

# Canales que no están en todos los sujetos
non_common_channels = all_channels - common_channels

# Para cada canal no común, ver en qué sujetos aparece
channel_presence = defaultdict(list)
for channel in non_common_channels:
    for subj, ch_set in sets_by_subject.items():
        if channel in ch_set:
            channel_presence[channel].append(subj)

# Mostrar resultados
print(f"✅ Canales presentes en TODOS los sujetos ({len(common_channels)}):")
print(sorted(common_channels))

print(f"\n⚠️ Canales NO presentes en todos los sujetos ({len(non_common_channels)}):")
for channel in sorted(non_common_channels):
    print(f"- {channel} está en: {channel_presence[channel]}")


✅ Canales presentes en TODOS los sujetos (273):
['MLC11-4304', 'MLC12-4304', 'MLC13-4304', 'MLC14-4304', 'MLC15-4304', 'MLC16-4304', 'MLC17-4304', 'MLC21-4304', 'MLC22-4304', 'MLC23-4304', 'MLC24-4304', 'MLC25-4304', 'MLC31-4304', 'MLC32-4304', 'MLC41-4304', 'MLC42-4304', 'MLC51-4304', 'MLC52-4304', 'MLC53-4304', 'MLC54-4304', 'MLC55-4304', 'MLC61-4304', 'MLC62-4304', 'MLC63-4304', 'MLF11-4304', 'MLF12-4304', 'MLF13-4304', 'MLF14-4304', 'MLF21-4304', 'MLF22-4304', 'MLF23-4304', 'MLF24-4304', 'MLF25-4304', 'MLF31-4304', 'MLF32-4304', 'MLF33-4304', 'MLF34-4304', 'MLF35-4304', 'MLF41-4304', 'MLF42-4304', 'MLF43-4304', 'MLF44-4304', 'MLF45-4304', 'MLF46-4304', 'MLF51-4304', 'MLF52-4304', 'MLF53-4304', 'MLF54-4304', 'MLF55-4304', 'MLF56-4304', 'MLF61-4304', 'MLF63-4304', 'MLF64-4304', 'MLF65-4304', 'MLF66-4304', 'MLF67-4304', 'MLO11-4304', 'MLO12-4304', 'MLO13-4304', 'MLO14-4304', 'MLO21-4304', 'MLO22-4304', 'MLO23-4304', 'MLO24-4304', 'MLO31-4304', 'MLO32-4304', 'MLO33-4304', 'MLO34-4304',